# Paper 3 — 03 · RD-DPO training

Trains one (anchor, condition, seed) tuple per run. Conditions:

| Condition           | Layers updated                              | Description                  |
|---------------------|---------------------------------------------|------------------------------|
| `sft`               | All linear modules, all blocks (LoRA r=16)  | SFT on chosen only           |
| `rd-dpo-k1/2/4/8`   | LoRA r=16 on top-k probe-selected blocks    | RD-DPO with k targeted blocks|
| `dpo-lora-all`      | All linear modules, all blocks (LoRA r=16)  | Standard LoRA-DPO baseline   |
| `dpo-full`          | Every parameter                             | Vanilla full-model DPO       |

**A100-40G runtime** (one epoch, ~12k pairs, seq 1024, effective batch 32):

| Anchor       | dpo-full | dpo-lora-all | rd-dpo-k4 |
|--------------|----------|--------------|-----------|
| Qwen-2.5-3B  | ~3 h     | ~2 h         | ~1 h      |
| Llama-3.2-3B | ~3 h     | ~2 h         | ~1 h      |
| Gemma-3-4B   | ~4 h     | ~2.5 h       | ~1.3 h    |

In [7]:
%%capture
# Pinned versions match METHOD_DESIGN §4 + requirements.txt. Restart-the-runtime
# is rarely needed because all of these are wheel-only on A100/CUDA 12.
!pip install -U \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    'peft>=0.13' \
    'trl>=0.11' \
    'datasets>=3.0' \
    'bitsandbytes>=0.44' \
    'torchao>=0.16' \
    huggingface_hub ipywidgets pyyaml -q


In [8]:
import os, json, gc, time, hashlib, math, sys
from pathlib import Path
from datetime import datetime
import torch

# --- Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper3-alignment")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")

PROBE_DIR    = DRIVE_ROOT / "data" / "probes"
PREFS_DIR    = DRIVE_ROOT / "data" / "preferences"
SPLITS_DIR   = DRIVE_ROOT / "data" / "splits"
ADAPTERS_DIR = DRIVE_ROOT / "adapters"
RESULTS_DIR  = DRIVE_ROOT / "results"
LOGS_DIR     = DRIVE_ROOT / "logs"
FIG_DIR      = DRIVE_ROOT / "figures"
for d in [PROBE_DIR, PREFS_DIR, SPLITS_DIR, ADAPTERS_DIR, RESULTS_DIR, LOGS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Paper 2 reuse: judges, llm_judge, holdout splits ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))

# --- A100 sanity + perf toggles ---
assert torch.cuda.is_available(), "GPU not available -- switch runtime to A100 GPU."
DEVICE_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {DEVICE_NAME}   VRAM: {VRAM_GB:.1f} GB   torch={torch.__version__}")
if "A100" not in DEVICE_NAME:
    print(f"WARNING: expected A100, got {DEVICE_NAME}. Re-tune BATCH_SIZE/grad-accum below.")

torch.set_float32_matmul_precision("high")          # TF32 for fp32 matmuls
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True               # autotune for fixed shapes

# --- HF / OpenRouter auth (set in Colab Secrets, not in code) ---
try:
    from google.colab import userdata
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    if os.environ.get("HF_TOKEN"):
        from huggingface_hub import login as _hf_login
        _hf_login(os.environ["HF_TOKEN"], add_to_git_credential=False)
    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY") or ""
except Exception as _e:
    print(f"(secrets not configured: {_e}; set HF_TOKEN / OPENROUTER_API_KEY in Colab secrets)")


def load_kwargs_for(family: str) -> dict:
    """A100-tuned dtype + attention impl per model family.

    Why bf16 everywhere: A100 has bf16 tensor cores at the same throughput as
    fp16, bf16 has the dynamic range of fp32 (no overflow on long sequences),
    and bf16 is the training dtype for all anchor families. Using fp16 here
    silently broke Gemma 3 in Paper 2 (953 empty-string outputs).
    """
    common = dict(torch_dtype=torch.bfloat16, device_map={"": 0})
    if family.startswith("gemma"):
        # Gemma 3 sliding-window attention is brittle with flash-attn-2.
        return {**common, "attn_implementation": "eager"}
    # PyTorch SDPA on A100 + bf16 is fast enough for our workloads.
    # flash-attn-2 was tried but its source-compile step costs ~15 min on
    # every cold Colab runtime, with negligible payoff for batch sizes <= 16.
    return {**common, "attn_implementation": "sdpa"}


# --- Paper 3 single-source-of-truth helpers ---
# short_of and family_of imported from src/prompts.py so the notebook
# convention stays aligned with notebook 02 and notebook 02b.
PROMPTS_SRC_DIR = DRIVE_ROOT / "src"
if not (PROMPTS_SRC_DIR / "prompts.py").exists():
    raise RuntimeError(
        f"src/prompts.py not found at {PROMPTS_SRC_DIR / 'prompts.py'}.\n"
        "Copy it from your local rosafety-align/src/prompts.py to that "
        "path in Drive, then re-run this cell."
    )
sys.path.insert(0, str(PROMPTS_SRC_DIR))
from prompts import (
    short_of, family_of,
    PROMPT_DIGESTS, CACHE_NAMESPACE_VERSION,
    verify_smoke_gate, SmokeGateNotPassed,
)

print("Bootstrap done.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA A100-SXM4-80GB   VRAM: 85.1 GB   torch=2.10.0+cu128


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Bootstrap done.


## Pre-flight gate

Verifies smoke gate is fresh, the augmented preferences file
(`preferences_v2.jsonl`) exists, and the probe artefact
(`selected_blocks.json`) exists for the configured anchor.
Training will not start without all three.


In [9]:
ANCHOR_PREFLIGHT = "Qwen/Qwen2.5-3B-Instruct"   # match cfg cell
short_pre  = short_of(ANCHOR_PREFLIGHT)

print("Pre-flight gate (PAPER3_PLAN section 15.7)")
print("-" * 72)
try:
    smoke = verify_smoke_gate(PREFS_DIR, ANCHOR_PREFLIGHT)
except SmokeGateNotPassed as e:
    raise SystemExit(f"\nPRE-FLIGHT FAIL\n{e}\n")

pairs_path_pre = PREFS_DIR / short_pre / "preferences_v2.jsonl"
if not pairs_path_pre.exists():
    raise SystemExit(
        f"\nPRE-FLIGHT FAIL\n"
        f"Augmented preferences not found at {pairs_path_pre}.\n"
        f"Run notebook 02b (Stage 4 augmentation) for {short_pre} first."
    )

probe_path_pre = PROBE_DIR / short_pre / "selected_blocks.json"
if not probe_path_pre.exists():
    raise SystemExit(
        f"\nPRE-FLIGHT FAIL\n"
        f"Probe artefacts not found at {probe_path_pre}.\n"
        f"Run notebook 01 (refusal-direction probe) for {short_pre} first."
    )

print(f"smoke ok               : {smoke['completed_at']}")
print(f"preferences_v2 found   : {pairs_path_pre}")
print(f"selected_blocks found  : {probe_path_pre}")
print(f"\nPRE-FLIGHT PASS. Anchor {short_pre!r} is ready for training.")


Pre-flight gate (PAPER3_PLAN section 15.7)
------------------------------------------------------------------------
smoke ok               : 2026-05-11T15:09:20+00:00
preferences_v2 found   : /content/drive/MyDrive/PhD/paper3-alignment/data/preferences/qwen2.5-3b/preferences_v2.jsonl
selected_blocks found  : /content/drive/MyDrive/PhD/paper3-alignment/data/probes/qwen2.5-3b/selected_blocks.json

PRE-FLIGHT PASS. Anchor 'qwen2.5-3b' is ready for training.


## Configuration

`SEED` cycles `{17, 1729, 65537}` per `configs/models.yaml`.

In [10]:
ANCHOR    = "Qwen/Qwen2.5-3B-Instruct"
CONDITION = "rd-dpo-k4-bal-b03"
SEED      = 17

# Locked at week 1 (METHOD §4.2). Per-anchor tuning of these is forbidden.
BETA              = 0.3       # bumped from 0.1 on 2026-05-13. The 0.1
                             # rebalanced run regressed jailbreak by 10pp
                             # and left toxicity 3.8pp below base; a higher
                             # beta keeps the policy closer to the base
                             # reference, muting drift caused by an
                             # imperfect data mix.
LR                = 5e-6
EPOCHS            = 3        # small dataset -> need more epochs
WARMUP_STEPS      = 10       # was 100; never fired at 5-6 steps/epoch
MAX_STEPS         = -1       # -1 = no cap (HF convention); EPOCHS rules.
                             # The first live run with MAX_STEPS=200 ran
                             # ~36 epochs on the 178-pair train set and
                             # over-fit (accuracy hit 1.0 by step 100).
                             # logging_steps=2 below: the second run only
                             # took 18 steps; with logging_steps=25 the
                             # trajectory was empty.
MAX_SEQ_LEN       = 1024
BATCH_PER_DEV     = 4
GRAD_ACCUM_STEPS  = 8       # effective batch 32
LORA_R, LORA_A    = 16, 32
LORA_DROPOUT      = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"]

short  = short_of(ANCHOR)
family = family_of(ANCHOR)

RUN_TAG = f"{short}__{CONDITION}__seed{SEED}"
RUN_DIR = ADAPTERS_DIR / RUN_TAG
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"run    : {RUN_TAG}")
print(f"adapter: {RUN_DIR}")


run    : qwen2.5-3b__rd-dpo-k4-bal-b03__seed17
adapter: /content/drive/MyDrive/PhD/paper3-alignment/adapters/qwen2.5-3b__rd-dpo-k4-bal-b03__seed17


## Resolve probe-selected blocks

For RD-DPO conditions only — `dpo-lora-all` / `dpo-full` / `sft` ignore.

In [11]:
selected_blocks = None
if CONDITION.startswith("rd-dpo-"):
    # CONDITION is e.g. 'rd-dpo-k4' or 'rd-dpo-k4-bal'. Strip the
    # 'rd-dpo-k' prefix and take the leading integer; anything after
    # the first non-digit (like '-bal') is a tag, not part of k.
    import re
    m = re.match(r"rd-dpo-k(\d+)", CONDITION)
    if not m:
        raise ValueError(f"could not parse k from CONDITION={CONDITION!r}")
    k = int(m.group(1))
    sel_path = PROBE_DIR / short / "selected_blocks.json"
    if not sel_path.exists():
        raise FileNotFoundError(
            f"Probe artefact not found at {sel_path}. "
            f"Run 01_refusal_probe.ipynb first."
        )
    sel = json.loads(sel_path.read_text())
    selected_blocks = sel[str(k)]
    print(f"selected blocks (k={k}): {selected_blocks}")
else:
    print("(non-RD condition; LoRA targets all blocks or full model)")

selected blocks (k=4): [31, 32, 33, 34]


## Load preferences + build dataset

In [12]:
from datasets import Dataset
from transformers import AutoTokenizer

pairs_path = PREFS_DIR / short / "preferences_v2.jsonl"
if not pairs_path.exists():
    raise FileNotFoundError(
        f"Preference dataset not found at {pairs_path}. "
        f"Run notebook 02 (Stage 1+2) and notebook 02b (Stage 4 augmentation) first."
    )
pairs = [json.loads(l) for l in open(pairs_path, encoding="utf-8")]
print(f"loaded {len(pairs)} pairs from {pairs_path}")

# --- 50/50 rebalance between refuse-side and help-side streams ---
# 2026-05-13 finding: at 53/47 help-vs-refuse on Qwen the recipe regressed
# toxicity refusal by 11.3 pp on the holdout split (results/...delta_vs_base.json).
# Hypothesis: the 99 over-refusal counter pairs out-pulled the 88 refuse-side
# pairs and the model generalised "do not produce RO apologetic-refusal"
# across the toxicity holdout. Down-sample the larger side to match the
# smaller side. Same probe, same beta, same epochs, same seed.
import random
REFUSE_SOURCES = {"core_s2", "xl"}
HELP_SOURCES   = {"overref"}
refuse = [r for r in pairs if r["meta"]["source"] in REFUSE_SOURCES]
helpp  = [r for r in pairs if r["meta"]["source"] in HELP_SOURCES]
other  = [r for r in pairs if r["meta"]["source"] not in REFUSE_SOURCES | HELP_SOURCES]
if other:
    raise ValueError(f"unexpected sources: {sorted({r['meta']['source'] for r in other})}")
n_target = min(len(refuse), len(helpp))
rng = random.Random(SEED)
refuse_kept = rng.sample(refuse, n_target) if len(refuse) > n_target else refuse
helpp_kept  = rng.sample(helpp,  n_target) if len(helpp)  > n_target else helpp
pairs = refuse_kept + helpp_kept
rng.shuffle(pairs)
from collections import Counter
src_counts = Counter(r["meta"]["source"] for r in pairs)
print(f"rebalanced to {len(pairs)} pairs ({n_target} refuse-side + {n_target} help-side)")
print(f"  source counts: {dict(src_counts)}")

tok = AutoTokenizer.from_pretrained(ANCHOR, padding_side="left")
if tok.pad_token is None: tok.pad_token = tok.eos_token

def format_row(r):
    prompt_chat = tok.apply_chat_template(
        [{"role": "user", "content": r["prompt"]}],
        tokenize=False, add_generation_prompt=True,
    )
    return {"prompt": prompt_chat, "chosen": r["chosen"], "rejected": r["rejected"]}

ds_full   = Dataset.from_list([format_row(r) for r in pairs])
splits    = ds_full.train_test_split(test_size=0.05, seed=SEED)
ds_train  = splits["train"]; ds_eval = splits["test"]
print({"train": len(ds_train), "eval": len(ds_eval)})


loaded 187 pairs from /content/drive/MyDrive/PhD/paper3-alignment/data/preferences/qwen2.5-3b/preferences_v2.jsonl
rebalanced to 176 pairs (88 refuse-side + 88 help-side)
  source counts: {'core_s2': 65, 'overref': 88, 'xl': 23}


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'train': 167, 'eval': 9}


## Build LoRA target spec

PEFT's `target_modules` accepts a list of module name suffixes or a regex. For
RD-DPO we restrict by **block index** via a regex matching `layers.{i}.…`.

In [13]:
from peft import LoraConfig

if CONDITION == "dpo-full":
    lora_config = None
elif CONDITION in ("sft", "dpo-lora-all"):
    lora_config = LoraConfig(
        r=LORA_R, lora_alpha=LORA_A, lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES, bias="none", task_type="CAUSAL_LM",
    )
elif CONDITION.startswith("rd-dpo-"):
    block_pat = "|".join(str(b) for b in selected_blocks)
    modules_re = "|".join(LORA_TARGET_MODULES)
    target_re = rf"^.*\.layers\.({block_pat})\.(?:.*\.)?(?:{modules_re})$"
    lora_config = LoraConfig(
        r=LORA_R, lora_alpha=LORA_A, lora_dropout=LORA_DROPOUT,
        target_modules=target_re, bias="none", task_type="CAUSAL_LM",
    )
else:
    raise ValueError(f"unknown condition {CONDITION!r}")
print("lora_config:", lora_config)

lora_config: LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules='^.*\\.layers\\.(31|32|33|34)\\.(?:.*\\.)?(?:q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$', exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


## Load model + train

A100 footprint:
- bf16 throughout (Paper 2 R-Gemma-3 lesson: never fp16).
- Gradient checkpointing on.
- TRL precomputes reference log-probs on dev (avoids carrying the reference model forward through every step).
- For `dpo-full`, expect ~38 GB peak; for LoRA conditions, ~16-22 GB.

In [14]:
from transformers import AutoModelForCausalLM, set_seed
from trl import DPOConfig, DPOTrainer

set_seed(SEED)

model = AutoModelForCausalLM.from_pretrained(ANCHOR, **load_kwargs_for(family))
model.config.use_cache = False
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

if CONDITION == "sft":
    from trl import SFTTrainer, SFTConfig
    sft_cfg = SFTConfig(
        output_dir=str(RUN_DIR), num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_PER_DEV,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LR, lr_scheduler_type="cosine", warmup_steps=WARMUP_STEPS,
        bf16=True, gradient_checkpointing=True,
        logging_steps=2, save_steps=250, eval_steps=100,
        seed=SEED, report_to=["none"],
        dataset_text_field="chosen", max_length=MAX_SEQ_LEN,
        max_steps=MAX_STEPS,
    )
    trainer = SFTTrainer(
        model=model, args=sft_cfg, peft_config=lora_config,
        train_dataset=ds_train, eval_dataset=ds_eval, processing_class=tok,
    )
else:
    dpo_cfg = DPOConfig(
        output_dir=str(RUN_DIR), num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_PER_DEV,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LR, lr_scheduler_type="cosine", warmup_steps=WARMUP_STEPS,
        bf16=True, gradient_checkpointing=True,
        logging_steps=2, save_steps=250, eval_steps=100,
        seed=SEED, report_to=["none"],
        beta=BETA, loss_type="sigmoid",
        max_length=MAX_SEQ_LEN,
        precompute_ref_log_probs=True,
        max_steps=MAX_STEPS,
    )
    trainer = DPOTrainer(
        model=model, args=dpo_cfg, peft_config=lora_config,
        train_dataset=ds_train, eval_dataset=ds_eval, processing_class=tok,
    )

print("starting training...")
t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f"training done in {elapsed/60:.1f} min")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

Computing reference log probs for eval dataset:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


starting training...


Step,Training Loss
2,0.697558
4,0.723663
6,0.768298
8,0.709083
10,0.706497
12,0.699779
14,0.688857
16,0.702767
18,0.656852


training done in 1.7 min


## Save adapter + run-meta

In [15]:
trainer.save_model(str(RUN_DIR))
peak_mem = torch.cuda.max_memory_allocated() / 1e9

(RUN_DIR / "run_meta.json").write_text(json.dumps({
    "run_tag": RUN_TAG, "anchor": ANCHOR, "condition": CONDITION, "seed": SEED,
    "selected_blocks": selected_blocks,
    "preference_dataset": str(pairs_path),
    "n_pairs": len(pairs),
    "pair_source_counts": dict(src_counts),
    "training_compute": {
        "wallclock_seconds": round(elapsed, 1),
        "peak_gpu_memory_gb": round(peak_mem, 2),
        "device": DEVICE_NAME,
    },
    "hyperparams": {
        "beta": BETA, "lr": LR, "epochs": EPOCHS,
        "lora_r": LORA_R, "lora_alpha": LORA_A,
        "max_seq_len": MAX_SEQ_LEN,
        "per_device_train_batch_size": BATCH_PER_DEV,
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    },
    "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
}, indent=2))
print(f"saved adapter + meta -> {RUN_DIR}")

/tmp/ipykernel_1424/67487776.py:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",


saved adapter + meta -> /content/drive/MyDrive/PhD/paper3-alignment/adapters/qwen2.5-3b__rd-dpo-k4-bal-b03__seed17


## Batch run: Llama + Gemma at the rebalanced beta=0.1 recipe

Independent of the Qwen-only cells above. Trains the two remaining
anchors back-to-back so the operator can leave Colab unattended for
~10 minutes total. Same probe per anchor (loaded from
`data/probes/<short>/selected_blocks.json`), same hyperparameters
as the rebalanced Qwen run that produced the least-bad delta-vs-base.

Each anchor saves to its own adapter folder; the cell is a no-op
for an anchor that already has its run_meta.json on disk, so safe
to re-execute.


In [16]:
import gc, json, random, re, time
from collections import Counter
from datetime import datetime
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import DPOConfig, DPOTrainer

# ----- locked recipe (mirrors the Qwen rebalanced run) -----
# --- defensive: ensure prompts.py is importable when run standalone ---
# (no-op if bootstrap already ran in this kernel)
import sys as _sys
from pathlib import Path as _Path
_PROMPTS_SRC = _Path("/content/drive/MyDrive/PhD/paper3-alignment/src")
if _PROMPTS_SRC.exists() and str(_PROMPTS_SRC) not in _sys.path:
    _sys.path.insert(0, str(_PROMPTS_SRC))

from prompts import ANCHORS as BATCH_ANCHORS
BATCH_CONDITION = "rd-dpo-k4-bal-e6-mid"
BATCH_SEED      = 17

BATCH_BETA              = 0.1
BATCH_LR                = 5e-6
BATCH_EPOCHS            = 6
BATCH_WARMUP_STEPS      = 2
BATCH_MAX_STEPS         = -1
BATCH_MAX_SEQ_LEN       = 1024
BATCH_BATCH_PER_DEV     = 4
BATCH_GRAD_ACCUM_STEPS  = 8
BATCH_LORA_R            = 16
BATCH_LORA_A            = 32
BATCH_LORA_DROPOUT      = 0.05
BATCH_LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                             "gate_proj", "up_proj", "down_proj"]

REFUSE_SOURCES = {"core_s2", "xl"}
HELP_SOURCES   = {"overref"}


def _rebalance(pairs, seed):
    refuse = [r for r in pairs if r["meta"]["source"] in REFUSE_SOURCES]
    helpp  = [r for r in pairs if r["meta"]["source"] in HELP_SOURCES]
    other  = [r for r in pairs if r["meta"]["source"] not in REFUSE_SOURCES | HELP_SOURCES]
    if other:
        raise ValueError(f"unexpected sources: {sorted({r['meta']['source'] for r in other})}")
    n_target = min(len(refuse), len(helpp))
    rng = random.Random(seed)
    refuse_kept = rng.sample(refuse, n_target) if len(refuse) > n_target else refuse
    helpp_kept  = rng.sample(helpp,  n_target) if len(helpp)  > n_target else helpp
    out = refuse_kept + helpp_kept
    rng.shuffle(out)
    return out, n_target


def _train_one(anchor):
    short_a  = short_of(anchor)
    family_a = family_of(anchor)
    run_tag  = f"{short_a}__{BATCH_CONDITION}__seed{BATCH_SEED}"
    run_dir  = ADAPTERS_DIR / run_tag
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / "run_meta.json").exists():
        print(f"[{anchor}] run_meta.json already present at {run_dir}; skipping (delete to force rerun).")
        return

    print(f"\n=== training {anchor} -> {run_tag} ===")

    # Resolve probe-selected blocks
    m = re.match(r"rd-dpo-k(\d+)", BATCH_CONDITION)
    if not m:
        raise ValueError(f"could not parse k from {BATCH_CONDITION!r}")
    k = int(m.group(1))
    sel_path = PROBE_DIR / short_a / "selected_blocks.json"
    if BATCH_CONDITION.endswith("-mid") or "-mid-" in BATCH_CONDITION:
        sel_path = PROBE_DIR / short_a / "selected_blocks_mid.json"
    if not sel_path.exists():
        raise FileNotFoundError(f"probe artefact not found at {sel_path}; run notebook 01 first")
    selected_blocks = json.loads(sel_path.read_text())[str(k)]
    print(f"  probe blocks (k={k}): {selected_blocks}")

    # Load + rebalance preferences
    pairs_path = PREFS_DIR / short_a / "preferences_v2.jsonl"
    if not pairs_path.exists():
        raise FileNotFoundError(f"{pairs_path} missing; run notebook 02 + 02b first")
    pairs = [json.loads(l) for l in open(pairs_path, encoding="utf-8")]
    n_loaded = len(pairs)
    pairs, n_target = _rebalance(pairs, BATCH_SEED)
    src_counts = Counter(r["meta"]["source"] for r in pairs)
    print(f"  loaded {n_loaded} -> rebalanced {len(pairs)} pairs ({n_target}+{n_target}); sources={dict(src_counts)}")

    # Tokenize + chat-template
    tok_a = AutoTokenizer.from_pretrained(anchor, padding_side="left")
    if tok_a.pad_token is None: tok_a.pad_token = tok_a.eos_token

    def _format(r):
        prompt_chat = tok_a.apply_chat_template(
            [{"role": "user", "content": r["prompt"]}],
            tokenize=False, add_generation_prompt=True,
        )
        return {"prompt": prompt_chat, "chosen": r["chosen"], "rejected": r["rejected"]}

    ds_full   = Dataset.from_list([_format(r) for r in pairs])
    ds_split  = ds_full.train_test_split(test_size=0.05, seed=BATCH_SEED)
    ds_train_a, ds_eval_a = ds_split["train"], ds_split["test"]
    print(f"  train={len(ds_train_a)}  eval={len(ds_eval_a)}")

    # LoRA target spec
    block_pat  = "|".join(str(b) for b in selected_blocks)
    modules_re = "|".join(BATCH_LORA_TARGET_MODULES)
    target_re  = rf"^.*\.layers\.({block_pat})\.(?:.*\.)?(?:{modules_re})$"
    lora_cfg = LoraConfig(
        r=BATCH_LORA_R, lora_alpha=BATCH_LORA_A, lora_dropout=BATCH_LORA_DROPOUT,
        target_modules=target_re, bias="none", task_type="CAUSAL_LM",
    )

    set_seed(BATCH_SEED)
    model_a = AutoModelForCausalLM.from_pretrained(anchor, **load_kwargs_for(family_a))
    model_a.config.use_cache = False
    if hasattr(model_a, "gradient_checkpointing_enable"):
        model_a.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False},
        )

    dpo_cfg = DPOConfig(
        output_dir=str(run_dir), num_train_epochs=BATCH_EPOCHS,
        per_device_train_batch_size=BATCH_BATCH_PER_DEV,
        gradient_accumulation_steps=BATCH_GRAD_ACCUM_STEPS,
        learning_rate=BATCH_LR, lr_scheduler_type="cosine",
        warmup_steps=BATCH_WARMUP_STEPS,
        bf16=True, gradient_checkpointing=True,
        logging_steps=2, save_steps=250, eval_steps=100,
        seed=BATCH_SEED, report_to=["none"],
        beta=BATCH_BETA, loss_type="sigmoid",
        max_length=BATCH_MAX_SEQ_LEN,
        precompute_ref_log_probs=True,
        max_steps=BATCH_MAX_STEPS,
    )
    trainer_a = DPOTrainer(
        model=model_a, args=dpo_cfg, peft_config=lora_cfg,
        train_dataset=ds_train_a, eval_dataset=ds_eval_a, processing_class=tok_a,
    )

    print("  starting training...")
    t0 = time.time()
    trainer_a.train()
    elapsed = time.time() - t0
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"  done in {elapsed/60:.1f} min  peak={peak_mem:.2f} GB")

    trainer_a.save_model(str(run_dir))
    (run_dir / "run_meta.json").write_text(json.dumps({
        "run_tag": run_tag, "anchor": anchor, "condition": BATCH_CONDITION, "seed": BATCH_SEED,
        "selected_blocks": selected_blocks,
        "preference_dataset": str(pairs_path),
        "n_pairs": len(pairs),
        "pair_source_counts": dict(src_counts),
        "training_compute": {
            "wallclock_seconds": round(elapsed, 1),
            "peak_gpu_memory_gb": round(peak_mem, 2),
            "device": DEVICE_NAME,
        },
        "hyperparams": {
            "beta": BATCH_BETA, "lr": BATCH_LR, "epochs": BATCH_EPOCHS,
            "lora_r": BATCH_LORA_R, "lora_alpha": BATCH_LORA_A,
            "max_seq_len": BATCH_MAX_SEQ_LEN,
            "per_device_train_batch_size": BATCH_BATCH_PER_DEV,
            "gradient_accumulation_steps": BATCH_GRAD_ACCUM_STEPS,
        },
        "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }, indent=2))
    print(f"  saved -> {run_dir}")

    # Free GPU before next anchor
    del model_a, trainer_a
    torch.cuda.empty_cache(); gc.collect()


for anchor in BATCH_ANCHORS:
    _train_one(anchor)
print("\nbatch training done.")


[Qwen/Qwen2.5-3B-Instruct] run_meta.json already present at /content/drive/MyDrive/PhD/paper3-alignment/adapters/qwen2.5-3b__rd-dpo-k4-bal-e6-mid__seed17; skipping (delete to force rerun).
[meta-llama/Llama-3.2-3B-Instruct] run_meta.json already present at /content/drive/MyDrive/PhD/paper3-alignment/adapters/llama-3.2-3b__rd-dpo-k4-bal-e6-mid__seed17; skipping (delete to force rerun).
[google/gemma-3-4b-it] run_meta.json already present at /content/drive/MyDrive/PhD/paper3-alignment/adapters/gemma-3-4b__rd-dpo-k4-bal-e6-mid__seed17; skipping (delete to force rerun).

batch training done.


## LR sweep: Qwen + Llama at LR in {1e-5, 2e-5, 5e-5}

Tests whether Qwen and Llama can be trained on the same
rebalanced 176-200 preference pairs by raising the LR. The
default recipe (LR=5e-6) left both at the random baseline
while Gemma converged. If at LR=1e-5 or 2e-5 either anchor
trains to margins > 0.1, the recipe is fine and the gap
between Gemma and the others was just LR.

Six adapters total. Saves to
`adapters/<short>__rd-dpo-k4-bal-e6-lr{tag}__seed17/`.
Skips an anchor that already has its run_meta.json.

Probe selection: top-of-net (`selected_blocks.json`). The
mid-depth ablation rejected the depth hypothesis, so we go
back to the high-score top-of-net selection.


In [17]:
# --- defensive: ensure prompts.py is importable when run standalone ---
import sys as _sys
from pathlib import Path as _Path
_PROMPTS_SRC = _Path("/content/drive/MyDrive/PhD/paper3-alignment/src")
if _PROMPTS_SRC.exists() and str(_PROMPTS_SRC) not in _sys.path:
    _sys.path.insert(0, str(_PROMPTS_SRC))

import gc, json, random, re, time
from collections import Counter
from datetime import datetime
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import DPOConfig, DPOTrainer

# Sweep configuration: only Qwen + Llama (Gemma already trained, no point retesting)
SWEEP_ANCHORS = [
    "Qwen/Qwen2.5-3B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
]
SWEEP_LRS = [
    (1e-5, "1e5"),
    (2e-5, "2e5"),
    (5e-5, "5e5"),
]

SWEEP_SEED  = 17
SWEEP_BETA  = 0.1
SWEEP_EPOCHS  = 6
SWEEP_WARMUP  = 2
SWEEP_MAX_SEQ = 1024
SWEEP_BATCH   = 4
SWEEP_GA      = 8
SWEEP_LORA_R  = 16
SWEEP_LORA_A  = 32
SWEEP_LORA_DROPOUT = 0.05
SWEEP_LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"]

REFUSE_SOURCES = {"core_s2", "xl"}
HELP_SOURCES   = {"overref"}


def _rebalance(pairs, seed):
    refuse = [r for r in pairs if r["meta"]["source"] in REFUSE_SOURCES]
    helpp  = [r for r in pairs if r["meta"]["source"] in HELP_SOURCES]
    other  = [r for r in pairs if r["meta"]["source"] not in REFUSE_SOURCES | HELP_SOURCES]
    if other:
        raise ValueError(f"unexpected sources: {sorted({r['meta']['source'] for r in other})}")
    n_target = min(len(refuse), len(helpp))
    rng = random.Random(seed)
    refuse_kept = rng.sample(refuse, n_target) if len(refuse) > n_target else refuse
    helpp_kept  = rng.sample(helpp,  n_target) if len(helpp)  > n_target else helpp
    out = refuse_kept + helpp_kept
    rng.shuffle(out)
    return out, n_target


def _train_one(anchor, lr, lr_tag):
    short_a  = short_of(anchor)
    family_a = family_of(anchor)
    condition = f"rd-dpo-k4-bal-e6-lr{lr_tag}"
    run_tag   = f"{short_a}__{condition}__seed{SWEEP_SEED}"
    run_dir   = ADAPTERS_DIR / run_tag
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / "run_meta.json").exists():
        print(f"[{anchor} @ lr={lr:.0e}] {run_dir.name} already exists; skipping.")
        return

    print(f"\n=== training {anchor} @ lr={lr:.0e} -> {run_tag} ===")

    # Probe-selected blocks (top-of-net, original selection)
    sel_path = PROBE_DIR / short_a / "selected_blocks.json"
    if not sel_path.exists():
        raise FileNotFoundError(f"probe artefact not found at {sel_path}; run nb01 first")
    selected_blocks = json.loads(sel_path.read_text())["4"]
    print(f"  probe blocks: {selected_blocks}")

    # Load + rebalance preferences
    pairs_path = PREFS_DIR / short_a / "preferences_v2.jsonl"
    if not pairs_path.exists():
        raise FileNotFoundError(f"{pairs_path} missing; run nb02 + nb02b first")
    pairs = [json.loads(l) for l in open(pairs_path, encoding="utf-8")]
    n_loaded = len(pairs)
    pairs, n_target = _rebalance(pairs, SWEEP_SEED)
    src_counts = Counter(r["meta"]["source"] for r in pairs)
    print(f"  loaded {n_loaded} -> rebalanced {len(pairs)} pairs ({n_target}+{n_target}); sources={dict(src_counts)}")

    tok_a = AutoTokenizer.from_pretrained(anchor, padding_side="left")
    if tok_a.pad_token is None: tok_a.pad_token = tok_a.eos_token

    def _format(r):
        prompt_chat = tok_a.apply_chat_template(
            [{"role": "user", "content": r["prompt"]}],
            tokenize=False, add_generation_prompt=True,
        )
        return {"prompt": prompt_chat, "chosen": r["chosen"], "rejected": r["rejected"]}

    ds_full   = Dataset.from_list([_format(r) for r in pairs])
    ds_split  = ds_full.train_test_split(test_size=0.05, seed=SWEEP_SEED)
    ds_train_a, ds_eval_a = ds_split["train"], ds_split["test"]
    print(f"  train={len(ds_train_a)}  eval={len(ds_eval_a)}")

    block_pat  = "|".join(str(b) for b in selected_blocks)
    modules_re = "|".join(SWEEP_LORA_TARGETS)
    target_re  = rf"^.*\.layers\.({block_pat})\.(?:.*\.)?(?:{modules_re})$"
    lora_cfg = LoraConfig(
        r=SWEEP_LORA_R, lora_alpha=SWEEP_LORA_A, lora_dropout=SWEEP_LORA_DROPOUT,
        target_modules=target_re, bias="none", task_type="CAUSAL_LM",
    )

    set_seed(SWEEP_SEED)
    model_a = AutoModelForCausalLM.from_pretrained(anchor, **load_kwargs_for(family_a))
    model_a.config.use_cache = False
    if hasattr(model_a, "gradient_checkpointing_enable"):
        model_a.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False},
        )

    dpo_cfg = DPOConfig(
        output_dir=str(run_dir), num_train_epochs=SWEEP_EPOCHS,
        per_device_train_batch_size=SWEEP_BATCH,
        gradient_accumulation_steps=SWEEP_GA,
        learning_rate=lr, lr_scheduler_type="cosine",
        warmup_steps=SWEEP_WARMUP,
        bf16=True, gradient_checkpointing=True,
        logging_steps=2, save_steps=250, eval_steps=100,
        seed=SWEEP_SEED, report_to=["none"],
        beta=SWEEP_BETA, loss_type="sigmoid",
        max_length=SWEEP_MAX_SEQ,
        precompute_ref_log_probs=True,
        max_steps=-1,
    )
    trainer_a = DPOTrainer(
        model=model_a, args=dpo_cfg, peft_config=lora_cfg,
        train_dataset=ds_train_a, eval_dataset=ds_eval_a, processing_class=tok_a,
    )

    print("  starting training...")
    t0 = time.time()
    trainer_a.train()
    elapsed = time.time() - t0
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"  done in {elapsed/60:.1f} min  peak={peak_mem:.2f} GB")

    trainer_a.save_model(str(run_dir))
    (run_dir / "run_meta.json").write_text(json.dumps({
        "run_tag": run_tag, "anchor": anchor, "condition": condition,
        "seed": SWEEP_SEED, "selected_blocks": selected_blocks,
        "preference_dataset": str(pairs_path),
        "n_pairs": len(pairs),
        "pair_source_counts": dict(src_counts),
        "training_compute": {
            "wallclock_seconds": round(elapsed, 1),
            "peak_gpu_memory_gb": round(peak_mem, 2),
            "device": DEVICE_NAME,
        },
        "hyperparams": {
            "beta": SWEEP_BETA, "lr": lr, "epochs": SWEEP_EPOCHS,
            "warmup_steps": SWEEP_WARMUP,
            "lora_r": SWEEP_LORA_R, "lora_alpha": SWEEP_LORA_A,
            "max_seq_len": SWEEP_MAX_SEQ,
            "per_device_train_batch_size": SWEEP_BATCH,
            "gradient_accumulation_steps": SWEEP_GA,
        },
        "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }, indent=2))
    print(f"  saved -> {run_dir}")

    del model_a, trainer_a
    torch.cuda.empty_cache(); gc.collect()


for anchor in SWEEP_ANCHORS:
    for lr, lr_tag in SWEEP_LRS:
        _train_one(anchor, lr, lr_tag)
print("\nLR sweep training done.")



=== training Qwen/Qwen2.5-3B-Instruct @ lr=1e-05 -> qwen2.5-3b__rd-dpo-k4-bal-e6-lr1e5__seed17 ===
  probe blocks: [31, 32, 33, 34]
  loaded 187 -> rebalanced 176 pairs (88+88); sources={'core_s2': 65, 'overref': 88, 'xl': 23}
  train=167  eval=9


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/167 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/167 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/9 [00:00<?, ? examples/s]

...

  starting training...


Step,Training Loss
2,0.702554
4,0.699580
6,0.673897
8,0.627416
10,0.540478
12,0.479583
14,0.406429
16,0.330553
18,0.316420
20,0.268226


  done in 3.3 min  peak=23.10 GB


/tmp/ipykernel_1424/1674771863.py:169: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",


  saved -> /content/drive/MyDrive/PhD/paper3-alignment/adapters/llama-3.2-3b__rd-dpo-k4-bal-e6-lr5e5__seed17

LR sweep training done.


## x4 expansion: train Qwen / Llama / Gemma on preferences_x4

Loops over `ANCHORS` and trains each on `preferences_x4.jsonl`
at the LR that worked for it (Gemma 5e-6, Qwen+Llama 2e-5).
Same probe (top-of-net), same beta=0.1, same epochs=6, same
rebalance step, same seed=17.

Adapter folder: `adapters/<short>__rd-dpo-k4-bal-e6-x4__seed17/`.
Skips an anchor whose `run_meta.json` already exists.


In [ ]:
# --- defensive: ensure prompts.py is importable when run standalone ---
import sys as _sys
from pathlib import Path as _Path
_PROMPTS_SRC = _Path("/content/drive/MyDrive/PhD/paper3-alignment/src")
if _PROMPTS_SRC.exists() and str(_PROMPTS_SRC) not in _sys.path:
    _sys.path.insert(0, str(_PROMPTS_SRC))

import gc, json, random, re, time
from collections import Counter
from datetime import datetime
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import DPOConfig, DPOTrainer

from prompts import ANCHORS

# Per-anchor working LR, established by the LR sweep (rd-dpo-k4-bal-e6-lr*).
PER_ANCHOR_LR = {
    "Qwen/Qwen2.5-3B-Instruct":          2e-5,
    "meta-llama/Llama-3.2-3B-Instruct":  2e-5,
    "google/gemma-3-4b-it":               5e-6,
}

X4_CONDITION = "rd-dpo-k4-bal-e6-x4"
X4_SEED = 17
X4_BETA = 0.1
X4_EPOCHS = 6
X4_WARMUP = 2
X4_BATCH = 4
X4_GA = 8
X4_MAX_SEQ = 1024
X4_LORA_R = 16
X4_LORA_A = 32
X4_LORA_DROPOUT = 0.05
X4_LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]

REFUSE_SOURCES = {"core_s2", "core_s2_ext", "xl"}
HELP_SOURCES   = {"overref"}


def _rebalance(pairs, seed):
    refuse = [r for r in pairs if r["meta"]["source"] in REFUSE_SOURCES]
    helpp  = [r for r in pairs if r["meta"]["source"] in HELP_SOURCES]
    other  = [r for r in pairs if r["meta"]["source"] not in REFUSE_SOURCES | HELP_SOURCES]
    if other:
        raise ValueError(f"unexpected sources: {sorted({r['meta']['source'] for r in other})}")
    n_target = min(len(refuse), len(helpp))
    rng = random.Random(seed)
    refuse_kept = rng.sample(refuse, n_target) if len(refuse) > n_target else refuse
    helpp_kept  = rng.sample(helpp,  n_target) if len(helpp)  > n_target else helpp
    out = refuse_kept + helpp_kept
    rng.shuffle(out)
    return out, n_target


def _train_one(anchor):
    short_a  = short_of(anchor)
    family_a = family_of(anchor)
    lr = PER_ANCHOR_LR[anchor]
    run_tag = f"{short_a}__{X4_CONDITION}__seed{X4_SEED}"
    run_dir = ADAPTERS_DIR / run_tag
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / "run_meta.json").exists():
        print(f"[{anchor}] run_meta.json already exists; skipping.")
        return

    print(f"\n=== x4 training {anchor} @ lr={lr:.0e} -> {run_tag} ===")

    sel_path = PROBE_DIR / short_a / "selected_blocks.json"
    if not sel_path.exists():
        raise FileNotFoundError(f"probe artefact not found at {sel_path}")
    selected_blocks = json.loads(sel_path.read_text())["4"]

    pairs_path = PREFS_DIR / short_a / "preferences_x4.jsonl"
    if not pairs_path.exists():
        raise FileNotFoundError(f"{pairs_path} missing; run x4 batch in nb02 first")
    pairs = [json.loads(l) for l in open(pairs_path, encoding="utf-8")]
    n_loaded = len(pairs)
    pairs, n_target = _rebalance(pairs, X4_SEED)
    src_counts = Counter(r["meta"]["source"] for r in pairs)
    print(f"  loaded {n_loaded} -> rebalanced {len(pairs)} pairs ({n_target}+{n_target})")
    print(f"  source distribution: {dict(src_counts)}")

    tok_a = AutoTokenizer.from_pretrained(anchor, padding_side="left")
    if tok_a.pad_token is None: tok_a.pad_token = tok_a.eos_token

    def _format(r):
        prompt_chat = tok_a.apply_chat_template(
            [{"role": "user", "content": r["prompt"]}],
            tokenize=False, add_generation_prompt=True,
        )
        return {"prompt": prompt_chat, "chosen": r["chosen"], "rejected": r["rejected"]}

    ds_full   = Dataset.from_list([_format(r) for r in pairs])
    ds_split  = ds_full.train_test_split(test_size=0.05, seed=X4_SEED)
    ds_train_a, ds_eval_a = ds_split["train"], ds_split["test"]
    print(f"  train={len(ds_train_a)}  eval={len(ds_eval_a)}")

    block_pat  = "|".join(str(b) for b in selected_blocks)
    modules_re = "|".join(X4_LORA_TARGETS)
    target_re  = rf"^.*\.layers\.({block_pat})\.(?:.*\.)?(?:{modules_re})$"
    lora_cfg = LoraConfig(
        r=X4_LORA_R, lora_alpha=X4_LORA_A, lora_dropout=X4_LORA_DROPOUT,
        target_modules=target_re, bias="none", task_type="CAUSAL_LM",
    )

    set_seed(X4_SEED)
    model_a = AutoModelForCausalLM.from_pretrained(anchor, **load_kwargs_for(family_a))
    model_a.config.use_cache = False
    if hasattr(model_a, "gradient_checkpointing_enable"):
        model_a.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False},
        )

    dpo_cfg = DPOConfig(
        output_dir=str(run_dir), num_train_epochs=X4_EPOCHS,
        per_device_train_batch_size=X4_BATCH,
        gradient_accumulation_steps=X4_GA,
        learning_rate=lr, lr_scheduler_type="cosine",
        warmup_steps=X4_WARMUP,
        bf16=True, gradient_checkpointing=True,
        logging_steps=2, save_steps=250, eval_steps=100,
        seed=X4_SEED, report_to=["none"],
        beta=X4_BETA, loss_type="sigmoid",
        max_length=X4_MAX_SEQ,
        precompute_ref_log_probs=True,
        max_steps=-1,
    )
    trainer_a = DPOTrainer(
        model=model_a, args=dpo_cfg, peft_config=lora_cfg,
        train_dataset=ds_train_a, eval_dataset=ds_eval_a, processing_class=tok_a,
    )

    print("  starting training...")
    t0 = time.time()
    trainer_a.train()
    elapsed = time.time() - t0
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"  done in {elapsed/60:.1f} min  peak={peak_mem:.2f} GB")

    trainer_a.save_model(str(run_dir))
    (run_dir / "run_meta.json").write_text(json.dumps({
        "run_tag": run_tag, "anchor": anchor, "condition": X4_CONDITION,
        "seed": X4_SEED, "selected_blocks": selected_blocks,
        "preference_dataset": str(pairs_path),
        "n_pairs": len(pairs),
        "pair_source_counts": dict(src_counts),
        "training_compute": {
            "wallclock_seconds": round(elapsed, 1),
            "peak_gpu_memory_gb": round(peak_mem, 2),
            "device": DEVICE_NAME,
        },
        "hyperparams": {
            "beta": X4_BETA, "lr": lr, "epochs": X4_EPOCHS,
            "warmup_steps": X4_WARMUP,
            "lora_r": X4_LORA_R, "lora_alpha": X4_LORA_A,
            "max_seq_len": X4_MAX_SEQ,
            "per_device_train_batch_size": X4_BATCH,
            "gradient_accumulation_steps": X4_GA,
        },
        "saved_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }, indent=2))
    print(f"  saved -> {run_dir}")

    del model_a, trainer_a
    torch.cuda.empty_cache(); gc.collect()


for anchor in ANCHORS:
    _train_one(anchor)
print("\nx4 batch training done.")
